# Parte 2: CNN + Boosting para clasificación de frutas

Dataset: Fruits360 de Kaggle
Clases: Apple, Banana, Avocado

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('Kaggle configurado')

In [ ]:
!mkdir -p fruits_data
%cd fruits_data
!kaggle datasets download -d moltean/fruits
!unzip -q fruits.zip
%cd ..

print('Dataset descargado')

In [ ]:
!pip install -q tensorflow opencv-python xgboost lightgbm scikit-learn pandas numpy

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
import time
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
dataset_path = None
for root, dirs, files in os.walk('fruits_data'):
    if 'Training' in dirs and 'Test' in dirs:
        dataset_path = root
        break

if not dataset_path:
    raise FileNotFoundError('Dataset no encontrado')

train_dir = Path(dataset_path) / 'Training'
test_dir = Path(dataset_path) / 'Test'

print(f'Dataset en: {dataset_path}')
print(f'Training existe: {train_dir.exists()}')
print(f'Test existe: {test_dir.exists()}')

In [ ]:
def cargar_datos(ruta, clases, tamaño=128):
    imagenes = []
    etiquetas = []
    
    for idx, clase in enumerate(clases):
        clase_dir = Path(ruta) / clase
        if not clase_dir.exists():
            continue
        
        for archivo in clase_dir.glob('*.jpg'):
            img = cv2.imread(str(archivo))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (tamaño, tamaño))
            img = img / 255.0
            
            imagenes.append(img)
            etiquetas.append(idx)
    
    return np.array(imagenes), np.array(etiquetas)

clases = ['Apple', 'Banana', 'Avocado']
tamaño_img = 128

X_train, y_train = cargar_datos(train_dir, clases, tamaño_img)
X_test, y_test = cargar_datos(test_dir, clases, tamaño_img)

print(f'Datos cargados:')
print(f'  Train: {X_train.shape}')
print(f'  Test: {X_test.shape}')

In [ ]:
modelo = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(tamaño_img, tamaño_img, 3)),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),
    
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(clases), activation='softmax')
])

modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('Modelo CNN creado')

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

print('Entrenando CNN...')
inicio = time.time()

historial = modelo.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=30,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

tiempo_cnn = time.time() - inicio
print(f'CNN entrenada en {tiempo_cnn:.1f}s')

In [ ]:
loss_test, acc_cnn = modelo.evaluate(X_test, y_test, verbose=0)
y_pred_cnn = np.argmax(modelo.predict(X_test, verbose=0), axis=1)

prec_cnn = precision_score(y_test, y_pred_cnn, average='weighted')
rec_cnn = recall_score(y_test, y_pred_cnn, average='weighted')
f1_cnn = f1_score(y_test, y_pred_cnn, average='weighted')

print(f'CNN - Accuracy: {acc_cnn:.4f}')
print(f'CNN - Precision: {prec_cnn:.4f}')
print(f'CNN - Recall: {rec_cnn:.4f}')
print(f'CNN - F1: {f1_cnn:.4f}')

In [ ]:
extractor = models.Model(
    inputs=modelo.input,
    outputs=modelo.layers[-2].output
)

embeddings_train = extractor.predict(X_train, verbose=0)
embeddings_test = extractor.predict(X_test, verbose=0)

print(f'Embeddings: {embeddings_train.shape}')

In [ ]:
def extraer_features(imagenes):
    features = []
    for img in imagenes:
        gray = cv2.cvtColor((img * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        brillo = np.mean(gray)
        brillo_std = np.std(gray)
        
        sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        bordes = np.sqrt(sobel_x**2 + sobel_y**2)
        bordes_media = np.mean(bordes)
        bordes_std = np.std(bordes)
        
        canny = cv2.Canny(gray, 100, 200)
        canny_media = np.mean(canny)
        
        features.append([brillo, brillo_std, bordes_media, bordes_std, canny_media])
    
    return np.array(features)

features_train = extraer_features(X_train)
features_test = extraer_features(X_test)

print(f'Features: {features_train.shape}')

In [ ]:
X_combined_train = np.hstack([embeddings_train, features_train])
X_combined_test = np.hstack([embeddings_test, features_test])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_combined_train)
X_test_scaled = scaler.transform(X_combined_test)

print(f'Features totales: {X_train_scaled.shape[1]}')
print(f'  Embeddings: 128')
print(f'  Features: 5')

In [ ]:
print('Entrenando XGBoost...')
inicio = time.time()

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)

xgb_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    early_stopping_rounds=10,
    verbose=False
)

tiempo_xgb = time.time() - inicio
y_pred_xgb = xgb_model.predict(X_test_scaled)
acc_xgb = accuracy_score(y_test, y_pred_xgb)

prec_xgb = precision_score(y_test, y_pred_xgb, average='weighted')
rec_xgb = recall_score(y_test, y_pred_xgb, average='weighted')
f1_xgb = f1_score(y_test, y_pred_xgb, average='weighted')

print(f'XGBoost en {tiempo_xgb:.1f}s - Accuracy: {acc_xgb:.4f}')

In [ ]:
print('Entrenando LightGBM...')
inicio = time.time()

lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)

lgb_model.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    early_stopping_rounds=10,
    callbacks=[lgb.log_evaluation(period=0)]
)

tiempo_lgb = time.time() - inicio
y_pred_lgb = lgb_model.predict(X_test_scaled)
acc_lgb = accuracy_score(y_test, y_pred_lgb)

prec_lgb = precision_score(y_test, y_pred_lgb, average='weighted')
rec_lgb = recall_score(y_test, y_pred_lgb, average='weighted')
f1_lgb = f1_score(y_test, y_pred_lgb, average='weighted')

print(f'LightGBM en {tiempo_lgb:.1f}s - Accuracy: {acc_lgb:.4f}')

In [ ]:
print('\n' + '='*60)
print('RESULTADOS')
print('='*60)

resultados = pd.DataFrame({
    'Modelo': ['CNN', 'XGBoost', 'LightGBM'],
    'Accuracy': [acc_cnn, acc_xgb, acc_lgb],
    'Precision': [prec_cnn, prec_xgb, prec_lgb],
    'Recall': [rec_cnn, rec_xgb, rec_lgb],
    'F1-Score': [f1_cnn, f1_xgb, f1_lgb]
})

print(resultados.to_string(index=False))

mejor = resultados.loc[resultados['Accuracy'].idxmax()]
print(f'\nMejor modelo: {mejor["Modelo"]} con {mejor["Accuracy"]:.4f}')

## Análisis

### Qué modelo es mejor?
Los modelos de boosting (XGBoost, LightGBM) normalmente superan a CNN pura porque combinan embeddings profundos con características sintéticas extraídas de las imágenes.

### Por qué funcionan los embeddings?
Los embeddings de la CNN capturan características visuales complejas (texturas, formas). Las características sintéticas añaden información sobre brillo y bordes. Juntas, dan mucha más información que cualquiera de ellas sola.

### Interpretabilidad
XGBoost y LightGBM son más interpretables que CNN. Puedes ver qué característica importa más para cada predicción. CNN es una caja negra.

### Ventajas de cada uno
CNN: Maneja imágenes de cualquier tamaño, aprende automáticamente.
Boosting: Más rápido, interpretable, usa menos datos.

### Limitaciones
CNN requiere imágenes de tamaño fijo (128x128). Si cambias el tamaño, pierdes información. Boosting necesita características predefinidas.